# Compare computation times of online runs

Timing per trial from two `source: online` histories (`wasabi_online.py`), e.g. the same rig
with different settings.

**Latency** runs from the end of the trial window (IR crossing + `offset_ms + trial_ms`, 600 ms
by default) to the next stim decided, and is the sum of three steps:

- **fit delay**: from the window end first seen on disk to the start of fitting (`fit_delay_s`).
  Its last part is **wait**, the `next_trial` call that returned the trial (keypress: incl. `catch_up`).
- **fitting**: `observe_trial`, filter and refit.
- **acquisition**: scoring the candidates for the next stim.

Alongside it:

- **preprocess**: spike detection run by the background thread since the previous trial. It
  overlaps fitting and the trial itself, so it is compute load, not latency.
- **cycle**: time between consecutive stim decisions (`decided_at`), the whole loop incl. the trial.

Histories from before `fit_delay_s` was recorded get an estimated fit delay (see
`estimate_fit_delay`, marked ~ / est.); their preprocess is `load_s - wait_s`. The first trial / first stim include JIT compilation (and
trial 0's preprocess the threshold phase), so the medians leave them out.

In [ ]:
from pathlib import Path

DATA_DIR = Path("/home/bryan/mnt/cullen_data/Current Project Databases - NHP/2025 Cerebellum prosthesis/Bryan/wasabi_debug_files")
RUNS = None   # or {"label": "history_....npz", ...}; None: every history in DATA_DIR, labelled by its name's suffix

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if RUNS is None:
    files = sorted(DATA_DIR.glob("history_*.npz"))
    # history_doptimal_20261112_115725_intan_only.npz -> "intan_only"
    RUNS = {"_".join(f.stem.split("_")[4:]) or f.stem: f for f in files}

H = {name: dict(np.load(DATA_DIR / path)) for name, path in RUNS.items()}
THREAD_POLL_S = 0.1  # IntanStream.start poll_s: the detection thread's sleep when no new data
MAIN_POLL_S = 0.05   # wasabi_online's poll for the next trial with an IR trigger


def estimate_fit_delay(h):
    """Fit delay of a history from before it was recorded (IR trigger assumed), from:

    - the last detection pass: the pass that sees the window end processes the data written
      during the previous pass and the sleep after it. At detection load r (detection time /
      cycle time) a pass of duration d covers d + THREAD_POLL_S of data, so d = r (d + poll),
      d = poll r / (1 - r); the trial is ready when that pass ends.
    - the main loop's poll: on average half of MAIN_POLL_S until it looks again.
    - wait_s, measured: the next_trial call that returned the trial.
    """
    r = np.clip(h["preprocess_s"] / h["cycle_s"], 0, 0.95)
    return THREAD_POLL_S * r / (1 - r) + MAIN_POLL_S / 2 + h["wait_s"]


for name, h in H.items():
    n = len(h["choice"])
    h["cycle_s"] = np.diff(h["decided_at"])[:n]  # decision t -> decision t+1: trial t's whole loop
    h["fit_delay_estimated"] = "fit_delay_s" not in h
    if h["fit_delay_estimated"]:  # older histories: load_s = preprocess_s + wait_s, no fit delay
        h["preprocess_s"] = h["load_s"] - h["wait_s"]
        h["fit_delay_s"] = estimate_fit_delay(h)
    acquisition = np.full(n, np.nan)
    after = h["acquisition_s"][1:n + 1]  # acquisition_s[t + 1] is the decision after trial t
    acquisition[:len(after)] = after
    h["acquisition_next_s"] = acquisition
    h["latency_s"] = h["fit_delay_s"] + h["fitting_s"] + acquisition  # window end -> next stim decided
    note = "  (fit delay estimated: not recorded)" if h["fit_delay_estimated"] else ""
    print(f"{name:12s} {Path(RUNS[name]).name}: {n} trials{note}")

# Categorical slots of the validated default palette, fixed per run
COLORS = dict(zip(H, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))
INK, MUTED, GRID = "#1f1f1e", "#6b6a63", "#e4e3df"
plt.rcParams.update({
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.spines.top": False,
    "axes.spines.right": False, "axes.titlesize": 11, "axes.titlelocation": "left",
    "legend.frameon": False, "lines.linewidth": 1.6,
})

# (history key, title); acquisition has one more entry than trials (entry 0: the first stim)
PANELS = [
    ("fit_delay_s", "Fit delay (window end to fitting)"),
    ("fitting_s", "Fitting (observe_trial)"),
    ("acquisition_s", "Acquisition (next stim)"),
    ("latency_s", "Latency (window end to next stim)"),
    ("wait_s", "Wait (next_trial call, part of fit delay)"),
    ("preprocess_s", "Preprocess (background spike detection)"),
    ("cycle_s", "Cycle (decision to decision)"),
]


def median(key, h):
    """Median leaving out the first trial (acquisition: the first two decisions), which include JIT."""
    values = h[key][2:] if key == "acquisition_s" else h[key][1:]
    return np.nanmedian(values) if np.isfinite(values).any() else np.nan

## Summary

Medians over trials 2 onward (acquisition: stims 3 onward), plus the control phase.
`detect / cycle` is background detection time over the cycle: near 1, detection only just
keeps up with the recording.

In [ ]:
def sec(value, width, digits):
    """Seconds right-aligned in `width`, "-" where not recorded."""
    return f"{value:{width - 1}.{digits}f}s" if np.isfinite(value) else f"{'-':>{width}s}"


print(f"{'':12s} {'trials':>6s}  {'fit delay':>9s} {'wait':>6s} {'fitting':>7s} {'acq.':>6s} {'latency':>7s}  "
      f"{'preproc':>7s} {'cycle':>6s} {'detect/cycle':>12s}   {'control load':>12s} {'fit':>6s}")
for name, h in H.items():
    med = {key: median(key, h) for key, _ in PANELS}
    ratio = np.nanmedian(h["preprocess_s"][1:] / h["cycle_s"][1:])
    est = "~" if h["fit_delay_estimated"] else " "  # ~: estimated, not recorded
    print(f"{name:12s} {len(h['choice']):6d} {est}{sec(med['fit_delay_s'], 9, 3)} {med['wait_s']:5.3f}s "
          f"{med['fitting_s']:6.2f}s {med['acquisition_s']:5.3f}s{est}{sec(med['latency_s'], 7, 2)}  "
          f"{med['preprocess_s']:6.2f}s {med['cycle_s']:5.1f}s {ratio:12.2f}   "
          f"{float(h['control_load_s']):11.1f}s {float(h['control_fit_s']):5.1f}s")

## Latency per trial

The three steps from the end of trial t's window to the next stim, and their sum. Acquisition
is the decision made after trial t. Dashed lines with open markers are estimated fit delays
(histories from before `fit_delay_s` was recorded; see `estimate_fit_delay`), and so are the
latencies built on them.

In [ ]:
LATENCY_PANELS = [
    ("fit_delay_s", "Fit delay (window end to fitting)"),
    ("fitting_s", "Fitting (observe_trial)"),
    ("acquisition_next_s", "Acquisition (next stim)"),
    ("latency_s", "Total latency (window end to next stim)"),
]
fig, axes = plt.subplots(1, 4, figsize=(18, 3.8), constrained_layout=True)
for ax, (key, title) in zip(axes, LATENCY_PANELS):
    for name, h in H.items():
        estimated = h["fit_delay_estimated"] and key in ("fit_delay_s", "latency_s")
        ax.plot(
            np.arange(len(h[key])), h[key], "o--" if estimated else "o-", ms=3, color=COLORS[name],
            mfc="none" if estimated else COLORS[name], label=f"{name} (est.)" if estimated else name,
        )
    ax.set(title=title, xlabel="trial", ylabel="s")
    ax.set_ylim(bottom=0)
axes[0].legend()
plt.show()

## Median breakdown

Where each cycle goes. Fit delay, fitting and acquisition are the latency; the rest of the cycle
is the trial and waiting for the next IR crossing. Preprocess runs alongside all of it, so it is
drawn next to the cycle, not stacked into it. Estimated fit delays are labelled (est.).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 0.9 + 0.8 * len(H)), constrained_layout=True)
STACK = [("fit_delay_s", "fit delay", "#6b6a63"), ("fitting_s", "fitting", "#2a78d6"), ("acquisition_s", "acquisition", "#eda100")]
for i, (name, h) in enumerate(H.items()):
    left = 0.0
    for key, label, color in STACK:
        width = median(key, h)
        if key == "fit_delay_s" and h["fit_delay_estimated"]:
            label = "fit delay (est.)"
        ax.barh(i + 0.18, width, left=left, height=0.32, color=color, label=label)
        left += width
    ax.barh(i + 0.18, median("cycle_s", h) - left, left=left, height=0.32, color=GRID, label="trial and idle")
    ax.barh(i - 0.18, median("preprocess_s", h), height=0.32, color="#1baf7a", label="preprocess (background)")
ax.set_yticks(range(len(H)), list(H))
ax.invert_yaxis()
ax.set(xlabel="median s per trial")
ax.grid(axis="y", visible=False)
handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))  # one entry per label across runs
ax.legend(unique.values(), unique.keys(), loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=6, fontsize=8)
plt.show()